# Investment Mandate and Assumptions
**Kappa Foundation | BFIN WS2026 | Decision date: 31 December 2022**

The below information is the team's proposed assumptions for the investment strategy

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from IPython.display import display

# Run from the project folder (or a notebook folder directly inside it).
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "data").is_dir()), cwd)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

INITIAL_WEALTH = 12_000_000
REAL_2030_GOAL = 8_000_000
LIABILITIES = {2024: 2_400_000, 2027: 3_600_000}
DECISION_YEAR, TERMINAL_YEAR = 2022, 2030


## 1. Assumed inflation and purchasing-power objective

**Team assumption:** The baseline uses the Federal Reserve's **14 December 2022 Summary of Economic Projections** median PCE inflation figures (3.1% for 2023, 2.5% for 2024, 2.1% for 2025 and 2.0% as the longer-run rate). Therefore, we apply 2.0% inflation rate for the period from 2026 to 2030. We also check a persistent 3% inflation scenario from 2026 onward.

The USD 8 million goal is stated in **2022 purchasing-power dollars**, so the nominal 2030 goal is USD 8 million compounded by the assumed annual inflation factors.

Source: [Federal Reserve, December 2022 SEP](https://www.federalreserve.gov/monetarypolicy/fomcprojtabl20221214.htm).

In [15]:
INFLATION_BASE = {2023: .031, 2024: .025, 2025: .021, **{y: .02 for y in range(2026, 2031)}}
INFLATION_SCENARIOS = {
    "Baseline": INFLATION_BASE,
    "Persistent 3% from 2026": {y: (r if y <= 2025 else .03) for y, r in INFLATION_BASE.items()},
}

def nominal_target(path):
    return REAL_2030_GOAL * np.prod([1 + path[y] for y in range(2023, 2031)])

def ending_wealth(annual_return):
    wealth = INITIAL_WEALTH
    for year in range(2023, TERMINAL_YEAR + 1):
        wealth = wealth * (1 + annual_return) - LIABILITIES.get(year, 0)
    return wealth

funding_table = pd.DataFrame([
    {"Scenario": name,
     "Nominal 2030 goal (USD)": nominal_target(path),
     "Required constant annual return": brentq(lambda r: ending_wealth(r) - nominal_target(path), 0, 1)}
    for name, path in INFLATION_SCENARIOS.items()
])
display(funding_table.style.format({"Nominal 2030 goal (USD)": "${:,.0f}", "Required constant annual return": "{:.2%}"}))
funding_table.to_csv(TABLES_DIR / "T1_funding_assumptions.csv", index=False)


,Scenario,Nominal 2030 goal (USD),Required constant annual return
0,Baseline,"$9,530,136",4.20%
1,Persistent 3% from 2026,"$10,006,550",4.67%


## 2. Our proposed risk policy

The trustees expressed discomfort with losses approaching 15%, but did not specify a horizon, confidence level, or binding cap. **Our team operationalizes this constraint as a 1-year 95% Value at Risk (VaR) capped at 15%** of beginning-of-year portfolio value:

$$
\text{VaR}_{95\%, 1\text{y}} \le 15\%
$$

Equivalently, the probability of a one-year portfolio loss exceeding 15% must not exceed 5%:

$$
P(R_{p, 1\text{y}} \le -0.15) \le 0.05
$$

Under a standard normal-return approximation:

$$
\text{VaR}_{95\%, 1\text{y}} \approx 1.645\sigma_p - \mu_p \le 0.15
$$

The 95% confidence level is our modelling choice, and we will test whether the complete portfolio meets the proposed limit.

## 3. Risk-return preference assumption

To provide a preliminary quantitative comparison without running full optimization in Task 1, we evaluate candidate allocations using **mean-variance utility**:

$$
U = E(R) - \frac{A}{2}\sigma^2
$$

We compare A = 1, 3, 5 as low, medium and high assumed risk aversion. 

We selected **A = 3 as a moderate risk-aversion** assumption because Kappa Foundation must balance its mandatory funding obligations with its long-term growth objective. We will test A = 1 and A = 5 to assess the sensitivity of the recommended allocation.